<a href="https://colab.research.google.com/github/Govind243/Integrated-sensing-and-communication-VLC/blob/main/VLS_SENSING_4AP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
VLC NLoS Sensing Spectral Efficiency — Literature-Corrected Version
====================================================================
Channel model aligned with:
  [1] Abdelhady et al., IEEE TVT 2021 — Lambertian LOS: (m+1)*A*g*cos^m(φ)*cos(ψ)/(2π*d²)
  [2] Aboagye et al., IEEE COMST 2023 — OIRS NLoS: (m+1)*ρ*A*dA*cosᵐ(φ₁)*cos(β₁)*cos(φ₂)*cos(β₂)/(2π²*d1²*d2²)
  [3] Arfaoui et al., IEEE Trans 2021 — VLC noise: shot + thermal + RIN
  [4] Adaptive DRL for IRS Mirror Orientation (arxiv 2505.01818)

Key corrections vs v1:
  - NLoS formula: dot2 → dot2² (cos²φ on the mirror→user leg for specular OIRS)
  - NLoS power: uses full power_per_AP (not shared) since sensing is per-AP
  - LOS channel: explicit irradiance/incidence angle separation (same angle, ceiling APs)
  - FoV clipping properly applied to both LOS and NLOS
  - Output path: saves to local folder + shows plot in notebook environments
  - Mirror tracking for sensing: SMA, DDPG, Hybrid all reported
"""

import os, warnings, time, random, collections
import numpy as np
import matplotlib
matplotlib.use('Agg')           # headless backend — safe in notebooks & scripts
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim

# ════════════════════════════════════════════════════════════════════════════
# Output directory — works in both scripts and Colab/Jupyter
# ════════════════════════════════════════════════════════════════════════════
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__))
                          if '__file__' in dir() else os.getcwd(), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'vlc_nlos_sensing_results.png')

# ════════════════════════════════════════════════════════════════════════════
# System Parameters
# ════════════════════════════════════════════════════════════════════════════
user_counts          = np.arange(2, 9)     # 2 .. 8 users
ITERATIONS_PER_COUNT = 300                 # Monte Carlo per user count

room_dimensions  = np.array([5.0, 5.0, 3.0])   # L × W × H [m]
total_power      = 1.0                           # total optical power budget [W]

# ── VLC Optical Parameters ────────────────────────────────────────────────
FOV         = 60.0                         # receiver half-angle FoV [deg]
Area_Pd     = 1e-4                         # PD area [m²]
Resp_pd     = 0.53                         # PD responsivity [A/W]
B_VLC       = 40e6                         # bandwidth [Hz]
PSD_noise   = 1e-21                        # thermal noise PSD [W/Hz]
q_charge    = 1.602e-19                    # electron charge [C]
RIN_lin     = 10 ** (-120.0 / 10.0)       # RIN linear [-120 dB/Hz]
n_refr      = 1.5                          # refractive index
psi_c       = np.deg2rad(FOV)             # FoV half-angle [rad]
phi_half    = np.deg2rad(60.0)            # LED half-power semi-angle [rad]
m_lambert   = -np.log(2) / np.log(np.cos(phi_half))   # Lambertian order
g_conc      = n_refr**2 / np.sin(psi_c)**2             # concentrator gain

# Noise variance (thermal only — shot computed per-link)
sigma2_th   = PSD_noise * B_VLC           # thermal noise variance [W]

# 4 VLC APs at ceiling, quarter-room symmetric placement
AP_vlc = np.array([
    [room_dimensions[0] / 4,     room_dimensions[1] / 4,     room_dimensions[2]],
    [3 * room_dimensions[0] / 4, 3 * room_dimensions[1] / 4, room_dimensions[2]],
    [3 * room_dimensions[0] / 4, room_dimensions[1] / 4,     room_dimensions[2]],
    [room_dimensions[0] / 4,     3 * room_dimensions[1] / 4, room_dimensions[2]],
])
num_APs     = AP_vlc.shape[0]
power_per_AP = total_power / num_APs

# ── OIRS Parameters ───────────────────────────────────────────────────────
MIRROR_CAP = 25        # total OIRS mirror budget
RHO_OIRS   = 0.9       # mirror reflectivity
NUM_OIRS   = 25        # number of physical mirror elements (= MIRROR_CAP)

# ── DDPG Hyper-parameters ─────────────────────────────────────────────────
DDPG_WARMUP  = 200
DDPG_BATCH   = 32
DDPG_BUF     = 10_000
LR_ACTOR     = 1e-4
LR_CRITIC    = 1e-3
GAMMA        = 0.99
TAU          = 0.005
OU_THETA     = 0.15
OU_SIGMA     = 0.20
NOISE_DECAY  = 0.9995
NOISE_MIN    = 0.02

# State features per user (10):
#   SNR_LOS_dB | inv_SNR | 1/dist_wall | cos_theta | AP_onehot[4] | H_nlos_sum | SNR_nlos_est_dB
FEAT_PER_USER = 10

print(f"Room: {room_dimensions} m  |  {num_APs} VLC APs  |  power/AP = {power_per_AP:.3f} W")
print(f"OIRS: {NUM_OIRS} mirrors  |  budget = {MIRROR_CAP}  |  rho = {RHO_OIRS}")
print(f"Thermal noise var = {sigma2_th:.4e} W  |  FoV = {FOV}°  |  m_Lambert = {m_lambert:.3f}")
print(f"DDPG: warmup={DDPG_WARMUP}  batch={DDPG_BATCH}  lr_a={LR_ACTOR}  lr_c={LR_CRITIC}\n")
print(f"Output will be saved to: {OUTPUT_PATH}\n")


# ════════════════════════════════════════════════════════════════════════════
# Channel & SNR Helpers  (literature-accurate)
# ════════════════════════════════════════════════════════════════════════════

def los_channel_gain(users_pos):
    """
    LOS channel gain H_los[ap, u] per Lambertian model [1]:
        H = (m+1)*A_r*g_c * cos^m(φ) * cos(ψ) / (2π * d²)
    For ceiling APs and upward-facing PDs: φ (irradiance at LED) = ψ (incidence
    at PD) = θ, the angle between the AP-user vector and the vertical.
    FoV constraint: H=0 if ψ > psi_c.

    Returns: H (num_APs, nu), dist (num_APs, nu), cos_theta (num_APs, nu)
    """
    nu = users_pos.shape[1]
    H  = np.zeros((num_APs, nu))
    D  = np.zeros((num_APs, nu))
    CT = np.zeros((num_APs, nu))

    for ap in range(num_APs):
        diff  = users_pos - AP_vlc[ap, :, np.newaxis]      # (3, nu)
        d     = np.linalg.norm(diff, axis=0)
        d     = np.maximum(d, 1e-6)
        # cos(θ) from vertical — both irradiance angle and incidence angle
        # AP is at ceiling pointing downward; user PD points upward
        cos_t = np.clip((AP_vlc[ap, 2] - users_pos[2]) / d, 0.0, 1.0)
        in_fov = cos_t >= np.cos(psi_c)   # FoV mask
        D[ap]  = d
        CT[ap] = cos_t
        H[ap]  = np.where(
            in_fov,
            (m_lambert + 1) * Area_Pd * g_conc
            * cos_t**m_lambert * cos_t           # cos^m(φ)*cos(ψ), φ=ψ=θ
            / (2 * np.pi * d**4),
            0.0)
    return H, D, CT


def vlc_snr(photocurrent):
    """
    Electrical SNR at the PD given received photocurrent I_rx [A].
    Noise: shot (signal-dependent) + thermal + RIN.
    All per IEEE/IET VLC literature standards [3].

    SNR = I_rx² / (σ²_shot + σ²_thermal + σ²_RIN)
        σ²_shot    = 2 * q * I_rx * B          [signal-induced]
        σ²_thermal = PSD_noise * B
        σ²_RIN     = RIN * I_rx² * B
    """
    I    = np.maximum(photocurrent, 0.0)
    shot = 2 * q_charge * I * B_VLC
    rin  = RIN_lin * I**2 * B_VLC
    return I**2 / (shot + sigma2_th + rin + 1e-30)


def sensing_se(snr_linear):
    """
    Sensing spectral efficiency: SE = log₂(1 + SNR) [bps/Hz].
    Combines LOS + NLoS SNR linearly (incoherent power combining).
    """
    return np.log2(1.0 + np.maximum(snr_linear, 0.0))


def compute_se_per_user(ap_asgn, H_los, snr_nlos_boost):
    """
    Compute per-user sensing SE.
    ap_asgn      : (nu,) int array, AP index 0..3
    H_los        : (num_APs, nu) LOS channel gains
    snr_nlos_boost: (nu,) NLoS SNR contribution (linear) from OIRS
    Returns      : (nu,) SE in bps/Hz
    """
    nu  = len(ap_asgn)
    cnt = np.bincount(ap_asgn, minlength=num_APs).astype(float)
    cnt = np.where(cnt == 0, 1.0, cnt)
    se  = np.zeros(nu)
    for u in range(nu):
        ap     = ap_asgn[u]
        pwr    = power_per_AP / cnt[ap]        # shared power (OMA/TDMA)
        I_los  = Resp_pd * pwr * H_los[ap, u] # photocurrent from LOS
        snr_los = vlc_snr(I_los)
        snr_eff = snr_los + snr_nlos_boost[u]  # linear sum (incoherent combining)
        se[u]   = sensing_se(snr_eff)
    return se


# ════════════════════════════════════════════════════════════════════════════
# AP Assignment
# ════════════════════════════════════════════════════════════════════════════

def greedy_ap(H_los):
    """Assign each user to the AP with max LOS channel gain (argmax)."""
    return np.argmax(H_los, axis=0)


def ml_ap(H_los, D, CT):
    """
    ML-based AP assignment: Random Forest trained on geometric features.
    Falls back to greedy for tiny user counts.
    """
    labels = np.argmax(H_los, axis=0)       # greedy as training target
    nu = H_los.shape[1]
    if nu < 4:
        return labels
    feats = np.vstack([H_los, D, CT]).T     # (nu, 12)
    try:
        rf = RandomForestRegressor(n_estimators=20, max_depth=4, random_state=0)
        rf.fit(feats, labels)
        pred = np.round(rf.predict(feats)).astype(int)
        return np.clip(pred, 0, num_APs - 1)
    except Exception:
        return labels


# ════════════════════════════════════════════════════════════════════════════
# OIRS Environment
# ════════════════════════════════════════════════════════════════════════════

class OIRSEnv:
    """
    NLoS OIRS mirror geometry (static mesh built once at init).
    Per-episode: call reset(users_pos, ap_asgn, H_los) to update runtime state.

    NLoS channel model per [2]:
        H_nlos[u,w] = Σ_ap  (m+1)*ρ*A_r*dA * cosᵐ(φ_aw) * cos(β_aw) *
                               cos²(φ_wu) / (2π² * d_aw² * d_wu²)
    where:
        φ_aw = irradiance angle at mirror from AP (=angle to wall normal)
        β_aw = incidence angle at mirror from AP  (same as φ_aw for flat wall)
        φ_wu = irradiance angle at user from mirror (= cos dot2)
        β_wu = incidence angle at PD from mirror  (= cos dot2, specular)
    → user-side cosine factor is cos(φ_wu)*cos(β_wu) = dot2 * dot2 = dot2²
    """

    def __init__(self):
        self.num_oirs   = NUM_OIRS
        self.MIRROR_CAP = MIRROR_CAP
        self.rho        = RHO_OIRS
        self.m          = m_lambert
        self.dA_elem    = None        # mirror element area (set below)

        # ── Wall mirror grid on Y=0 plane, 7×7, keep NUM_OIRS closest to centre ─
        wall_res   = 7
        x_w = np.linspace(0, room_dimensions[0], wall_res)
        z_w = np.linspace(0, room_dimensions[2], wall_res)
        Xw, Zw = np.meshgrid(x_w, z_w)
        all_pts = np.vstack([Xw.ravel(), np.zeros(wall_res**2), Zw.ravel()])
        self.dA_elem  = (room_dimensions[0] / wall_res) * (room_dimensions[2] / wall_res)
        ctr  = np.array([room_dimensions[0] / 2, 0.0, room_dimensions[2] / 2])
        idx  = np.argsort(np.linalg.norm(all_pts - ctr[:, None], axis=0))
        self.wall_pts    = all_pts[:, idx[:self.num_oirs]]   # (3, nw)
        self.wall_normal = np.array([0.0, 1.0, 0.0])

        # ── Candidate mirror normals: 3 yaw × 3 roll = 9 orientations ─────────
        normals = []
        for yaw in np.deg2rad([-30, 0, 30]):
            for roll in np.deg2rad([-30, 0, 30]):
                Ry = np.array([[np.cos(yaw), -np.sin(yaw), 0],
                               [np.sin(yaw),  np.cos(yaw), 0],
                               [0, 0, 1]])
                Rr = np.array([[1, 0, 0],
                               [0, np.cos(roll), -np.sin(roll)],
                               [0, np.sin(roll),  np.cos(roll)]])
                normals.append(Ry @ Rr @ np.array([0.0, 1.0, 0.0]))
        self.cand_normals = np.array(normals)   # (9, 3)

        # ── Precompute static AP → mirror geometry ─────────────────────────────
        nw = self.num_oirs
        self.d_ap_mir   = np.zeros((num_APs, nw))   # d1 : AP-mirror distance
        self.cos_ap_mir = np.zeros((num_APs, nw))   # dot1 = cos(φ_aw) = cos(β_aw)
        self.valid_ap   = np.zeros((num_APs, nw), dtype=bool)

        for a, ap_pos in enumerate(AP_vlc):
            diff_am   = ap_pos[None, :] - self.wall_pts.T       # (nw, 3)
            d1        = np.linalg.norm(diff_am, axis=1)
            d1        = np.maximum(d1, 1e-6)
            cos1      = np.clip((diff_am / d1[:, None]) @ self.wall_normal, 0.0, None)
            self.d_ap_mir[a]   = d1
            self.cos_ap_mir[a] = cos1
            self.valid_ap[a]   = cos1 > 0.1

        # Rank mirrors by peak AP-incident power (used for allocation order)
        P_inc = np.zeros((num_APs, nw))
        for a in range(num_APs):
            d1   = self.d_ap_mir[a]
            cos1 = self.cos_ap_mir[a]
            v    = self.valid_ap[a]
            P_inc[a] = np.where(v,
                power_per_AP * (self.m + 1) * cos1**self.m / (2 * np.pi * d1**2),
                0.0)
        self.mirror_order = np.argsort(P_inc.max(axis=0))[::-1].copy()

        # ── Runtime state (populated by reset) ────────────────────────────────
        self.nu            = None
        self.users         = None    # (nu, 3)
        self.H_nlos_mat    = None    # (nu, nw) NLoS channel gain matrix
        self.SNR_los_dB    = None    # (nu,) LOS SNR in dB
        self.dist_wall     = None    # (nu,) Y-distance to OIRS wall
        self.cos_wall      = None    # (nu,) cosine of user-to-wall angle

    # ── reset ─────────────────────────────────────────────────────────────────
    def reset(self, users_pos, ap_asgn, H_los):
        """
        users_pos : (3, nu)
        ap_asgn   : (nu,) AP indices 0..3
        H_los     : (num_APs, nu) LOS channel gains
        """
        nu = users_pos.shape[1]
        self.nu    = nu
        self.users = users_pos.T   # (nu, 3)

        # LOS SNR per user (dB)
        cnt = np.bincount(ap_asgn, minlength=num_APs).astype(float)
        cnt = np.where(cnt == 0, 1.0, cnt)
        snr_los = np.zeros(nu)
        for u in range(nu):
            ap  = ap_asgn[u]
            pwr = power_per_AP / cnt[ap]
            I   = Resp_pd * pwr * H_los[ap, u]
            snr_los[u] = vlc_snr(I)
        self.SNR_los_dB = 10.0 * np.log10(snr_los + 1e-12)

        # Geometry to Y=0 wall
        self.dist_wall = np.maximum(self.users[:, 1], 1e-3)
        user_d = np.linalg.norm(self.users, axis=1)
        self.cos_wall  = np.clip(self.users[:, 1] / np.maximum(user_d, 1e-6), 0.0, 1.0)

        # Compute NLoS channel gain matrix (literature-accurate)
        self._build_nlos_matrix()

    # ── NLoS channel (corrected: dot2² on user leg per [2]) ───────────────────
    def _build_nlos_matrix(self):
        """
        H_nlos[u, w] = Σ_ap  K * cosᵐ(φ_aw) * cos(β_aw) * cos²(φ_wu)
                              / (2π² * d_aw² * d_wu²)
        where K = (m+1)*ρ*A_r*dA and cos(β_aw)=cos(φ_aw)=cos1 for flat wall,
        and cos(φ_wu)*cos(β_wu) = dot2² for specular mirror (φ_wu = β_wu).

        Uses full power_per_AP for NLoS (sensing channel, not shared with UL).
        """
        nw, nu = self.num_oirs, self.nu

        # Mirror → user geometry: diff2 (nu, nw, 3), d2 (nu, nw)
        diff2 = self.users[:, None, :] - self.wall_pts.T[None, :, :]  # (nu,nw,3)
        d2    = np.linalg.norm(diff2, axis=2)
        d2    = np.maximum(d2, 1e-6)
        dir2  = diff2 / d2[:, :, None]

        # dot2[c, u, w] = cos of angle between mirror normal (cand c) and user direction
        dot2_all = np.einsum('uwd,cd->cuw', dir2, self.cand_normals)   # (9,nu,nw)
        dot2_all = np.clip(dot2_all, 0.0, None)

        # Pass 1: find best orientation per mirror element (max sum over all users & APs)
        H_sum = np.zeros((9, nw))
        K     = (self.m + 1) * self.rho * Area_Pd * self.dA_elem   # scalar

        for a in range(num_APs):
            d1   = self.d_ap_mir[a]                              # (nw,)
            cos1 = self.cos_ap_mir[a]                            # (nw,)
            v    = self.valid_ap[a]                              # (nw,)
            # AP→mirror contribution factor (Lambertianⁿˣⁿⁿⁿ × cosine incidence)
            # cos^m(φ_aw)*cos(β_aw) = cos1^m * cos1 = cos1^(m+1)  [flat wall]
            cos1_factor = np.where(v, cos1**(self.m + 1), 0.0)  # (nw,)
            denom = 2.0 * np.pi**2 * d1[None, :]**2 * d2**2 + 1e-30  # (nu,nw)
            # user-side factor: dot2² (specular: incidence = reflection angle)
            both_v = (dot2_all > 0.1) & v[None, None, :]         # (9,nu,nw)
            H_c    = np.where(both_v,
                              K * power_per_AP * cos1_factor[None, None, :]
                              * dot2_all**2 / denom[None, :, :],
                              0.0)
            H_sum += H_c.sum(axis=1)   # sum over users → (9, nw)

        best_orient = np.argmax(H_sum, axis=0)                     # (nw,)
        dot2_best   = dot2_all[best_orient, :, np.arange(nw)].T   # (nu, nw)

        # Pass 2: per-user NLoS gain with best orientations, full AP power
        H_nlos = np.zeros((nu, nw))
        for a in range(num_APs):
            d1   = self.d_ap_mir[a]
            cos1 = self.cos_ap_mir[a]
            v    = self.valid_ap[a]
            cos1_factor = np.where(v, cos1**(self.m + 1), 0.0)
            denom = 2.0 * np.pi**2 * d1[None, :]**2 * d2**2 + 1e-30
            both_v = (dot2_best > 0.1) & v[None, :]
            H_nlos += np.where(both_v,
                               K * power_per_AP * cos1_factor[None, :]
                               * dot2_best**2 / denom,
                               0.0)
        self.H_nlos_mat = H_nlos   # (nu, nw)

    # ── NLoS SNR from a binary mirror assignment ───────────────────────────────
    def nlos_snr(self, A):
        """
        A : (nu, nw) binary mirror assignment
        Returns (nu,) linear NLoS SNR values.
        """
        P_nlos = (self.H_nlos_mat * A.astype(float)).sum(axis=1)  # received optical power
        I_nlos = Resp_pd * P_nlos                                   # photocurrent
        return vlc_snr(I_nlos)

    # ── SMA: SNR-threshold mirror allocation ──────────────────────────────────
    def sma_alloc(self):
        """
        Threshold-based Static Mirror Allocation (greedy, weakest-first).
        Returns: A (nu, nw) binary, mirrors_used (int)
        """
        nu  = self.nu
        req = np.zeros(nu, dtype=int)
        for u in range(nu):
            s = self.SNR_los_dB[u]
            if   s <  0:  req[u] = 5
            elif s < 15:  req[u] = 4
            elif s < 30:  req[u] = 3
            else:          req[u] = 1

        order = np.argsort(self.SNR_los_dB)   # weakest first
        A     = np.zeros((nu, self.num_oirs), dtype=int)
        pool  = 0

        for u in order:
            if pool >= self.MIRROR_CAP:
                break
            k = min(int(req[u]), self.MIRROR_CAP - pool)
            if k > 0:
                A[u, self.mirror_order[pool:pool + k]] = 1
                pool += k

        # Distribute any remaining budget (still weakest-first)
        while pool < self.MIRROR_CAP:
            prev_pool = pool
            for u in order:
                if pool < self.MIRROR_CAP:
                    A[u, self.mirror_order[pool]] = 1
                    pool += 1
                else:
                    break
            if pool == prev_pool:
                break   # no progress

        return A, pool

    # ── DDPG: weight-based mirror allocation ──────────────────────────────────
    def ddpg_alloc(self, weights):
        """
        Convert normalised DDPG weights → binary mirror allocation.
        weights : (nu,) positive, summing to 1.0
        Returns : A (nu, nw) binary, mirrors_used (int)
        """
        nu      = self.nu
        mirrors = np.floor(weights * self.MIRROR_CAP).astype(int)
        deficit = self.MIRROR_CAP - mirrors.sum()
        if deficit > 0:
            fracs   = weights * self.MIRROR_CAP - mirrors
            top_idx = np.argsort(fracs)[::-1][:deficit]
            mirrors[top_idx] += 1

        A    = np.zeros((nu, self.num_oirs), dtype=int)
        pool = 0
        for u in np.argsort(weights)[::-1]:
            if pool >= self.MIRROR_CAP:
                break
            k = min(int(mirrors[u]), self.MIRROR_CAP - pool)
            if k > 0:
                A[u, self.mirror_order[pool:pool + k]] = 1
                pool += k
        return A, pool

    # ── Hybrid: element-wise min(k_sma, k_ddpg) ──────────────────────────────
    def hybrid_alloc(self, weights):
        """
        Hybrid = min(SMA allocation, DDPG allocation) per user.
        Saved mirrors are intentionally NOT redistributed — this is the key
        metric showing the hybrid scheme's mirror efficiency.

        Returns: A (nu, nw) binary, mirrors_used (int), mirrors_saved (int)
        """
        nu = self.nu

        # SMA counts
        k_sma = np.zeros(nu, dtype=int)
        for u in range(nu):
            s = self.SNR_los_dB[u]
            if   s <  0:  k_sma[u] = 5
            elif s < 15:  k_sma[u] = 4
            elif s < 30:  k_sma[u] = 3
            else:          k_sma[u] = 1
        sma_sum = k_sma.sum()
        if sma_sum > self.MIRROR_CAP:
            k_sma = np.maximum(1, np.floor(k_sma * self.MIRROR_CAP / sma_sum).astype(int))

        # DDPG counts
        k_ddpg  = np.floor(weights * self.MIRROR_CAP).astype(int)
        deficit = self.MIRROR_CAP - k_ddpg.sum()
        if deficit > 0:
            fracs   = weights * self.MIRROR_CAP - k_ddpg
            top_idx = np.argsort(fracs)[::-1][:deficit]
            k_ddpg[top_idx] += 1

        # Intersection: element-wise min
        k_hyb      = np.minimum(k_ddpg, k_sma)
        user_order = np.argsort(weights)[::-1]

        A    = np.zeros((nu, self.num_oirs), dtype=int)
        pool = 0
        for u in user_order:
            if pool >= self.MIRROR_CAP:
                break
            k = min(int(k_hyb[u]), self.MIRROR_CAP - pool)
            if k > 0:
                A[u, self.mirror_order[pool:pool + k]] = 1
                pool += k

        mirrors_saved = self.MIRROR_CAP - pool
        return A, pool, mirrors_saved

    # ── DDPG state vector builder ──────────────────────────────────────────────
    def build_state(self, ap_asgn):
        """
        Flat state vector (nu × 10):
          [SNR_los_dB, inv_SNR, 1/dist_wall, cos_wall, AP_onehot×4,
           H_nlos_sum, SNR_nlos_sma_dB]
        """
        nu        = self.nu
        snr_lin   = 10 ** (self.SNR_los_dB / 10.0)
        inv_snr   = 1.0 / (snr_lin + 1e-6)
        inv_dist  = 1.0 / self.dist_wall

        ap_oh = np.zeros((nu, num_APs), dtype=float)
        ap_oh[np.arange(nu), np.clip(ap_asgn, 0, num_APs - 1)] = 1.0

        H_nlos_sum = self.H_nlos_mat.sum(axis=1)

        A_sma, _ = self.sma_alloc()
        snr_nlos_sma = self.nlos_snr(A_sma)
        snr_nlos_dB  = 10.0 * np.log10(snr_nlos_sma + 1e-12)

        features = np.column_stack([
            self.SNR_los_dB,   # (nu,)
            inv_snr,           # (nu,)
            inv_dist,          # (nu,)
            self.cos_wall,     # (nu,)
            ap_oh,             # (nu, 4)
            H_nlos_sum,        # (nu,)
            snr_nlos_dB,       # (nu,)
        ])                     # → (nu, 10)
        return features.astype(np.float32).ravel()


# ════════════════════════════════════════════════════════════════════════════
# DDPG Components
# ════════════════════════════════════════════════════════════════════════════

class OUNoise:
    def __init__(self, size):
        self.mu    = np.zeros(size)
        self.reset()

    def reset(self):
        self.state = self.mu.copy()

    def sample(self):
        dx = OU_THETA * (self.mu - self.state) + OU_SIGMA * np.random.randn(len(self.mu))
        self.state += dx
        return self.state


class ReplayBuffer:
    def __init__(self, maxsize=DDPG_BUF):
        self.buf  = []
        self.ptr  = 0
        self.maxsz = maxsize

    def push(self, s, a, r, s2, done):
        e = (s.copy(), a.copy(), float(r), s2.copy(), float(done))
        if len(self.buf) < self.maxsz:
            self.buf.append(e)
        else:
            self.buf[self.ptr] = e
        self.ptr = (self.ptr + 1) % self.maxsz

    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, r, s2, d = map(np.array, zip(*batch))
        return s, a, r.astype(np.float32), s2, d.astype(np.float32)

    def __len__(self):
        return len(self.buf)


class Actor(nn.Module):
    def __init__(self, sdim, adim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(sdim),
            nn.Linear(sdim, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 128),  nn.LayerNorm(128), nn.ReLU(),
            nn.Linear(128, adim), nn.Sigmoid(),
        )
        nn.init.uniform_(self.net[-2].weight, -3e-3, 3e-3)
        nn.init.uniform_(self.net[-2].bias,   -3e-3, 3e-3)

    def forward(self, x):
        return self.net(x)


class Critic(nn.Module):
    def __init__(self, sdim, adim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(sdim + adim),
            nn.Linear(sdim + adim, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 128),         nn.LayerNorm(128), nn.ReLU(),
            nn.Linear(128, 1),
        )
        nn.init.uniform_(self.net[-1].weight, -3e-3, 3e-3)
        nn.init.uniform_(self.net[-1].bias,   -3e-3, 3e-3)

    def forward(self, s, a):
        return self.net(torch.cat([s, a], dim=1))


class DDPGAgent:
    """One agent per user count (fixed state/action dims for that count)."""
    def __init__(self, sdim, adim):
        self.adim      = adim
        self.noise     = OUNoise(adim)
        self.noise_std = 0.20
        self.buf       = ReplayBuffer()

        self.actor    = Actor(sdim, adim)
        self.actor_t  = Actor(sdim, adim)
        self.critic   = Critic(sdim, adim)
        self.critic_t = Critic(sdim, adim)
        self.actor_t.load_state_dict(self.actor.state_dict())
        self.critic_t.load_state_dict(self.critic.state_dict())

        self.opt_a = optim.Adam(self.actor.parameters(),  lr=LR_ACTOR)
        self.opt_c = optim.Adam(self.critic.parameters(), lr=LR_CRITIC)

    def act(self, state, explore=True):
        s = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            a = self.actor(s).cpu().numpy()[0]
        if explore:
            a = a + self.noise.sample() * self.noise_std
        a = np.clip(a, 1e-6, 1.0)
        return a / a.sum()

    def train(self):
        if len(self.buf) < DDPG_BATCH:
            return
        s, a, r, s2, done = self.buf.sample(DDPG_BATCH)
        S  = torch.FloatTensor(s)
        A  = torch.FloatTensor(a)
        R  = torch.FloatTensor(r).unsqueeze(1)
        S2 = torch.FloatTensor(s2)
        D  = torch.FloatTensor(done).unsqueeze(1)

        with torch.no_grad():
            A2    = self.actor_t(S2)
            Q_tgt = R + GAMMA * (1 - D) * self.critic_t(S2, A2)

        c_loss = nn.MSELoss()(self.critic(S, A), Q_tgt)
        self.opt_c.zero_grad()
        c_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.opt_c.step()

        a_loss = -self.critic(S, self.actor(S)).mean()
        self.opt_a.zero_grad()
        a_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
        self.opt_a.step()

        for p, pt in zip(self.actor.parameters(),  self.actor_t.parameters()):
            pt.data.copy_(TAU * p.data + (1 - TAU) * pt.data)
        for p, pt in zip(self.critic.parameters(), self.critic_t.parameters()):
            pt.data.copy_(TAU * p.data + (1 - TAU) * pt.data)

        self.noise_std = max(NOISE_MIN, self.noise_std * NOISE_DECAY)


# ════════════════════════════════════════════════════════════════════════════
# Single Monte Carlo Iteration
# ════════════════════════════════════════════════════════════════════════════

def run_iteration(nu, env, agent):
    """
    One random user layout → compute SE for all 4 schemes,
    train DDPG, return results.
    """
    # Random user positions (users on floor plane at 0.85 m height)
    users_pos = np.array([
        np.random.uniform(0.5, room_dimensions[0] - 0.5, nu),
        np.random.uniform(0.5, room_dimensions[1] - 0.5, nu),
        np.full(nu, 0.85),
    ])   # (3, nu)

    H_los, D, CT = los_channel_gain(users_pos)

    asgn_greedy = greedy_ap(H_los)
    asgn_ml     = ml_ap(H_los, D, CT)

    # ── 1. LOS only (no OIRS) ─────────────────────────────────────────────
    se_los = compute_se_per_user(asgn_greedy, H_los, np.zeros(nu))

    # ── 2. SMA OIRS (greedy AP) ───────────────────────────────────────────
    env.reset(users_pos, asgn_greedy, H_los)
    A_sma, mir_sma = env.sma_alloc()
    se_sma = compute_se_per_user(asgn_greedy, H_los, env.nlos_snr(A_sma))

    # ── 3. DDPG OIRS (greedy AP) — get weights from greedy env ───────────
    state  = env.build_state(asgn_greedy)
    explore = len(agent.buf) < DDPG_WARMUP * 3
    weights = agent.act(state, explore=explore)

    A_ddpg, mir_ddpg = env.ddpg_alloc(weights)
    se_ddpg = compute_se_per_user(asgn_greedy, H_los, env.nlos_snr(A_ddpg))

    # ── 4. Hybrid OIRS (ML AP + SMA∩DDPG mirrors) ────────────────────────
    env.reset(users_pos, asgn_ml, H_los)
    A_hyb, mir_hyb, mir_saved = env.hybrid_alloc(weights)
    se_hyb = compute_se_per_user(asgn_ml, H_los, env.nlos_snr(A_hyb))

    # ── DDPG reward & experience replay ──────────────────────────────────
    jfi    = se_ddpg.sum()**2 / (nu * (se_ddpg**2).sum() + 1e-12)
    reward = 0.7 * se_ddpg.mean() / 5.0 + 0.3 * jfi
    state2 = env.build_state(asgn_ml)
    agent.buf.push(state, weights, reward, state2, 1.0)
    agent.train()

    return (se_los, se_sma, se_ddpg, se_hyb,
            mir_sma, mir_ddpg, mir_hyb, mir_saved)


# ════════════════════════════════════════════════════════════════════════════
# Main Monte Carlo Loop
# ════════════════════════════════════════════════════════════════════════════

n_uc = len(user_counts)

avg_SE_los  = np.zeros(n_uc)
avg_SE_sma  = np.zeros(n_uc)
avg_SE_ddpg = np.zeros(n_uc)
avg_SE_hyb  = np.zeros(n_uc)

avg_mir_sma   = np.zeros(n_uc)
avg_mir_ddpg  = np.zeros(n_uc)
avg_mir_hyb   = np.zeros(n_uc)
avg_mir_saved = np.zeros(n_uc)

se_per_mir_sma  = np.zeros(n_uc)
se_per_mir_ddpg = np.zeros(n_uc)
se_per_mir_hyb  = np.zeros(n_uc)

avg_jfi_los  = np.zeros(n_uc)
avg_jfi_sma  = np.zeros(n_uc)
avg_jfi_ddpg = np.zeros(n_uc)
avg_jfi_hyb  = np.zeros(n_uc)

env = OIRSEnv()

print("=" * 68)
print(f"{'Users':>6} | {'LOS SE':>8} | {'SMA SE':>8} | "
      f"{'DDPG SE':>8} | {'Hyb SE':>8} | {'M_Hyb':>5} | {'Saved':>5}")
print("-" * 68)

for ci, nu in enumerate(user_counts):
    sdim  = nu * FEAT_PER_USER
    adim  = nu
    agent = DDPGAgent(sdim, adim)

    acc = collections.defaultdict(list)
    t0  = time.perf_counter()

    for _ in range(ITERATIONS_PER_COUNT):
        (se_los, se_sma, se_ddpg, se_hyb,
         mir_sma, mir_ddpg, mir_hyb, mir_saved) = run_iteration(nu, env, agent)

        acc['los'].append(se_los.mean())
        acc['sma'].append(se_sma.mean())
        acc['ddpg'].append(se_ddpg.mean())
        acc['hyb'].append(se_hyb.mean())
        acc['m_sma'].append(mir_sma)
        acc['m_ddpg'].append(mir_ddpg)
        acc['m_hyb'].append(mir_hyb)
        acc['m_saved'].append(mir_saved)

        def jfi(s): return s.sum()**2 / (nu * (s**2).sum() + 1e-12)
        acc['jfi_los'].append(jfi(se_los))
        acc['jfi_sma'].append(jfi(se_sma))
        acc['jfi_ddpg'].append(jfi(se_ddpg))
        acc['jfi_hyb'].append(jfi(se_hyb))

    elapsed = time.perf_counter() - t0

    avg_SE_los[ci]  = np.mean(acc['los'])
    avg_SE_sma[ci]  = np.mean(acc['sma'])
    avg_SE_ddpg[ci] = np.mean(acc['ddpg'])
    avg_SE_hyb[ci]  = np.mean(acc['hyb'])

    avg_mir_sma[ci]   = np.mean(acc['m_sma'])
    avg_mir_ddpg[ci]  = np.mean(acc['m_ddpg'])
    avg_mir_hyb[ci]   = np.mean(acc['m_hyb'])
    avg_mir_saved[ci] = np.mean(acc['m_saved'])

    se_per_mir_sma[ci]  = avg_SE_sma[ci]  / max(avg_mir_sma[ci],  1e-3)
    se_per_mir_ddpg[ci] = avg_SE_ddpg[ci] / max(avg_mir_ddpg[ci], 1e-3)
    se_per_mir_hyb[ci]  = avg_SE_hyb[ci]  / max(avg_mir_hyb[ci],  1e-3)

    avg_jfi_los[ci]  = np.mean(acc['jfi_los'])
    avg_jfi_sma[ci]  = np.mean(acc['jfi_sma'])
    avg_jfi_ddpg[ci] = np.mean(acc['jfi_ddpg'])
    avg_jfi_hyb[ci]  = np.mean(acc['jfi_hyb'])

    print(f"{nu:>6} | {avg_SE_los[ci]:>8.4f} | {avg_SE_sma[ci]:>8.4f} | "
          f"{avg_SE_ddpg[ci]:>8.4f} | {avg_SE_hyb[ci]:>8.4f} | "
          f"{avg_mir_hyb[ci]:>5.1f} | {avg_mir_saved[ci]:>5.1f}  [{elapsed:.1f}s]")

print("=" * 68)


# ════════════════════════════════════════════════════════════════════════════
# Plots
# ════════════════════════════════════════════════════════════════════════════

uc = user_counts
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle(
    "VLC NLoS Sensing: OIRS Mirror Allocation Comparison\n"
    "(Literature-Accurate Channel Model: Lambertian LOS + Specular OIRS NLoS)",
    fontsize=13, fontweight='bold')

# ── Plot 1: Average Sensing SE vs Users ──────────────────────────────────
ax = axes[0, 0]
ax.plot(uc, avg_SE_los,  'k--o', label='LOS Only (Greedy)',  lw=1.8, ms=6)
ax.plot(uc, avg_SE_sma,  'b-s',  label='SMA OIRS (Greedy)',  lw=2,   ms=7)
ax.plot(uc, avg_SE_ddpg, 'r-^',  label='DDPG OIRS (Greedy)', lw=2,   ms=7)
ax.plot(uc, avg_SE_hyb,  'g-D',  label='Hybrid OIRS (ML AP)',lw=2.5, ms=8)
ax.set_xlabel('Number of Users')
ax.set_ylabel('Avg Sensing SE (bps/Hz/user)')
ax.set_title('Sensing Spectral Efficiency vs Users')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
ax.set_xticks(uc)

# ── Plot 2: Mirrors Used ─────────────────────────────────────────────────
ax = axes[0, 1]
ax.axhline(MIRROR_CAP, color='gray', ls=':', lw=1.5, label=f'Budget ({MIRROR_CAP})')
ax.plot(uc, avg_mir_sma,  'b-s',  label='SMA',    lw=2,   ms=7)
ax.plot(uc, avg_mir_ddpg, 'r-^',  label='DDPG',   lw=2,   ms=7)
ax.plot(uc, avg_mir_hyb,  'g-D',  label='Hybrid', lw=2.5, ms=8)
ax.fill_between(uc, avg_mir_hyb, np.full_like(uc, MIRROR_CAP, dtype=float),
                alpha=0.12, color='green', label='Saved by Hybrid')
ax.set_xlabel('Number of Users')
ax.set_ylabel('Avg Mirrors Used for Sensing')
ax.set_title('OIRS Mirrors Used per Scheme')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
ax.set_xticks(uc)
ax.set_ylim(0, MIRROR_CAP + 3)

# ── Plot 3: Grouped Bar — Mirror Usage Comparison ────────────────────────
ax = axes[0, 2]
w = 0.25
ax.bar(uc - w, avg_mir_sma,  w, label='SMA',    color='steelblue', alpha=0.85)
ax.bar(uc,     avg_mir_ddpg, w, label='DDPG',   color='tomato',    alpha=0.85)
ax.bar(uc + w, avg_mir_hyb,  w, label='Hybrid', color='seagreen',  alpha=0.85)
ax.axhline(MIRROR_CAP, color='gray', ls='--', lw=1.5, label=f'Budget ({MIRROR_CAP})')
ax.set_xlabel('Number of Users')
ax.set_ylabel('Avg Mirrors Used')
ax.set_title('Mirror Usage Comparison (Grouped Bar)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(uc)

# ── Plot 4: SE per Mirror (Efficiency) ───────────────────────────────────
ax = axes[1, 0]
ax.plot(uc, se_per_mir_sma,  'b-s',  label='SMA',    lw=2,   ms=7)
ax.plot(uc, se_per_mir_ddpg, 'r-^',  label='DDPG',   lw=2,   ms=7)
ax.plot(uc, se_per_mir_hyb,  'g-D',  label='Hybrid', lw=2.5, ms=8)
ax.set_xlabel('Number of Users')
ax.set_ylabel('SE per Mirror (bps/Hz/mirror)')
ax.set_title('Mirror Efficiency: Sensing SE per Mirror')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
ax.set_xticks(uc)

# ── Plot 5: Jain's Fairness Index ────────────────────────────────────────
ax = axes[1, 1]
ax.plot(uc, avg_jfi_los,  'k--o', label='LOS',    lw=1.8, ms=6)
ax.plot(uc, avg_jfi_sma,  'b-s',  label='SMA',    lw=2,   ms=7)
ax.plot(uc, avg_jfi_ddpg, 'r-^',  label='DDPG',   lw=2,   ms=7)
ax.plot(uc, avg_jfi_hyb,  'g-D',  label='Hybrid', lw=2.5, ms=8)
ax.set_xlabel('Number of Users')
ax.set_ylabel("Jain's Fairness Index")
ax.set_title('Sensing Fairness vs Users')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
ax.set_xticks(uc)
ax.set_ylim(0, 1.05)

# ── Plot 6: Mirrors Saved by Hybrid ──────────────────────────────────────
ax = axes[1, 2]
pct = 100.0 * avg_mir_saved / MIRROR_CAP
ax.bar(uc, avg_mir_saved, color='seagreen', alpha=0.8, label='Mirrors Saved')
for x, s, p in zip(uc, avg_mir_saved, pct):
    ax.text(x, s + 0.15, f'{p:.0f}%', ha='center', va='bottom', fontsize=9,
            fontweight='bold')
ax.set_xlabel('Number of Users')
ax.set_ylabel('Avg Mirrors Saved vs Full Budget')
ax.set_title('Mirror Savings by Hybrid (SMA∩DDPG)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(uc)
ax.set_ylim(0, MIRROR_CAP + 3)

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved → {OUTPUT_PATH}")

# ════════════════════════════════════════════════════════════════════════════
# Summary Table
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 95)
print(f"{'Nu':>4} | {'LOS_SE':>7} | {'SMA_SE':>7} | {'DDP_SE':>7} | {'Hyb_SE':>7} "
      f"| {'M_SMA':>5} | {'M_DDP':>5} | {'M_Hyb':>5} | {'Saved':>5} | {'Eff_Hyb':>8}")
print("-" * 95)
for ci, nu in enumerate(user_counts):
    print(f"{nu:>4} | {avg_SE_los[ci]:>7.4f} | {avg_SE_sma[ci]:>7.4f} | "
          f"{avg_SE_ddpg[ci]:>7.4f} | {avg_SE_hyb[ci]:>7.4f} "
          f"| {avg_mir_sma[ci]:>5.1f} | {avg_mir_ddpg[ci]:>5.1f} "
          f"| {avg_mir_hyb[ci]:>5.1f} | {avg_mir_saved[ci]:>5.1f} "
          f"| {se_per_mir_hyb[ci]:>8.5f}")
print("=" * 95)
print("""
Column legend:
  LOS_SE   = No OIRS, greedy AP assignment            [bps/Hz/user]
  SMA_SE   = SNR-threshold SMA mirrors, greedy AP     [bps/Hz/user]
  DDP_SE   = DDPG-weighted mirrors, greedy AP         [bps/Hz/user]
  Hyb_SE   = Hybrid (SMA∩DDPG), ML AP assignment     [bps/Hz/user]
  M_SMA    = Avg mirrors used by SMA
  M_DDP    = Avg mirrors used by DDPG
  M_Hyb    = Avg mirrors used by Hybrid (< MIRROR_CAP by design)
  Saved    = Avg mirrors saved vs full budget (MIRROR_CAP - M_Hyb)
  Eff_Hyb  = Hybrid sensing SE per mirror used        [bps/Hz/mirror]
""")

Room: [5. 5. 3.] m  |  4 VLC APs  |  power/AP = 0.250 W
OIRS: 25 mirrors  |  budget = 25  |  rho = 0.9
Thermal noise var = 4.0000e-14 W  |  FoV = 60.0°  |  m_Lambert = 1.000
DDPG: warmup=200  batch=32  lr_a=0.0001  lr_c=0.001

Output will be saved to: /content/outputs/vlc_nlos_sensing_results.png

 Users |   LOS SE |   SMA SE |  DDPG SE |   Hyb SE | M_Hyb | Saved
--------------------------------------------------------------------
     2 |   1.8763 |   2.1131 |   2.1328 |   1.9340 |   8.3 |  16.7  [3.4s]
     3 |   1.7035 |   1.8431 |   1.8586 |   1.7594 |  12.3 |  12.7  [3.1s]
     4 |   1.4557 |   1.5728 |   1.5704 |   0.9917 |  17.7 |   7.3  [12.8s]
     5 |   1.2531 |   1.3494 |   1.3564 |   0.8440 |  18.8 |   6.2  [12.7s]
     6 |   1.1000 |   1.1778 |   1.1711 |   0.8169 |  19.3 |   5.7  [12.6s]
     7 |   0.9517 |   1.0361 |   1.0122 |   0.6904 |  18.1 |   6.9  [12.8s]
     8 |   0.8296 |   0.9037 |   0.8830 |   0.6101 |  19.4 |   5.6  [12.6s]

Plot saved → /content/outputs/vlc_